# Paraphrasing Analysis

In [7]:
import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

cluster_labels = pd.read_csv("cluster_labels.csv")
sample_labels = pd.read_csv("sample_labels.csv")

df_cluster = pd.read_json("sampled_from_defan_clustered_1000_NOMATH_8B_temp0.8_topp0.95_maxtokens32.jsonl",lines=True)
df_sample = pd.read_json("sampled_from_defan_1000_NOMATH_8B_temp0.8_topp0.95_maxtokens32.jsonl",lines=True)

df_cluster["label"] = cluster_labels["label"]
df_sample["label"] = sample_labels["label"]

# ------------------------------------------------------------
# basic sanity checks
# ------------------------------------------------------------
for name, df in [("cluster", df_cluster), ("sample", df_sample)]:
    print(f"\n{name.upper()} DATA")
    print("-" * 50)
    print("rows:", len(df))
    print("unique class_id:", df["class_id"].nunique())
    print("label distribution:")
    print(df["label"].value_counts(normalize=True).sort_index())

    counts = df["class_id"].value_counts().sort_index()
    print("\nresponses per class_id:")
    print(counts.value_counts().sort_index())

# expected:
# - 1000 unique class_id
# - exactly 15 rows per class_id

# ------------------------------------------------------------
# load embeddings
# ------------------------------------------------------------
E_cluster = np.load("cluster_embeddings.npy")
E_sample = np.load("sample_embeddings.npy")

print("raw embedding shapes:")
print("E_cluster:", E_cluster.shape)
print("E_sample: ", E_sample.shape)


# ------------------------------------------------------------
# group 15 rows at a time
# ------------------------------------------------------------
def group_embeddings(E, group_size=15):
    E = np.asarray(E)
    if E.ndim != 2:
        raise ValueError(f"Expected 2D array, got {E.shape}")

    n_rows, d = E.shape
    if n_rows % group_size != 0:
        raise ValueError(f"{n_rows} rows is not divisible by {group_size}")

    return E.reshape(n_rows // group_size, group_size, d)

E_cluster_grouped = group_embeddings(E_cluster, group_size=15)
E_sample_grouped = group_embeddings(E_sample, group_size=15)

print("\ngrouped embedding shapes:")
print("E_cluster_grouped:", E_cluster_grouped.shape)
print("E_sample_grouped: ", E_sample_grouped.shape)

# ------------------------------------------------------------
# helper functions
# ------------------------------------------------------------
def safe_normalize_rows(X, eps=1e-12):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.maximum(norms, eps)
    return X / norms


def safe_normalize_vec(x, eps=1e-12):
    n = np.linalg.norm(x)
    if n < eps:
        return x
    return x / n


def semantic_entropy_from_labels(labels):
    counts = np.bincount(labels)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return -(probs * np.log2(probs)).sum()


def cosine_matrices(E, renormalize=True):
    if renormalize:
        E = np.stack([safe_normalize_rows(X) for X in E], axis=0)
    return E @ E.transpose(0, 2, 1)


def cluster_labels(E, tau=0.85):
    """
    Agglomerative clustering with cosine similarity threshold tau.
    distance threshold = 1 - tau
    """
    out = []

    for X in E:
        model = AgglomerativeClustering(
            metric="cosine",
            linkage="average",
            distance_threshold=1 - tau,
            n_clusters=None,
        )
        out.append(model.fit_predict(X))

    return out

# ------------------------------------------------------------
# baseline feature computation
# features:
#   H_sem
#   D_cos
#   D_pair
#   n_clusters
#   sigma2_S
#   D_cos_var
# ------------------------------------------------------------
def compute_baseline_features(E, cluster_tau=0.85, renormalize=True):
    if renormalize:
        E = np.stack([safe_normalize_rows(X) for X in E], axis=0)

    S = cosine_matrices(E, renormalize=False)
    cluster_list = cluster_labels(E, tau=cluster_tau)

    n_q, n_resp, _ = E.shape
    iu = np.triu_indices(n_resp, k=1)

    rows = []

    for i in range(n_q):
        X = E[i]
        Si = S[i]
        labels = cluster_list[i]

        # pairwise similarities and distances
        sims = Si[iu]
        dists = 1.0 - sims

        D_pair = dists.mean()
        sigma2_S = sims.var()

        # centroid distances
        c = safe_normalize_vec(X.mean(axis=0))
        centroid_dists = 1.0 - (X @ c)

        D_cos = centroid_dists.mean()
        D_cos_var = centroid_dists.var()

        # clustering features
        H_sem = semantic_entropy_from_labels(labels)
        n_clusters = len(np.unique(labels))

        rows.append({
            "H_sem": H_sem,
            "D_cos": D_cos,
            "D_pair": D_pair,
            "n_clusters": n_clusters,
            "sigma2_S": sigma2_S,
            "D_cos_var": D_cos_var,
        })

    return pd.DataFrame(rows)

# ------------------------------------------------------------
# compute grouped feature matrices
# ------------------------------------------------------------
X_cluster = compute_baseline_features(E_cluster_grouped, cluster_tau=0.85, renormalize=True)
X_sample = compute_baseline_features(E_sample_grouped, cluster_tau=0.85, renormalize=True)

print("feature matrix shapes:")
print("X_cluster:", X_cluster.shape)
print("X_sample: ", X_sample.shape)

print("\ncolumns:")
print(X_cluster.columns.tolist())

# ------------------------------------------------------------
# compute grouped feature matrices
# ------------------------------------------------------------
X_cluster = compute_baseline_features(E_cluster_grouped, cluster_tau=0.85, renormalize=True)
X_sample = compute_baseline_features(E_sample_grouped, cluster_tau=0.85, renormalize=True)

print("feature matrix shapes:")
print("X_cluster:", X_cluster.shape)
print("X_sample: ", X_sample.shape)

print("\ncolumns:")
print(X_cluster.columns.tolist())

# ------------------------------------------------------------
# construct y = 1 iff majority (>=8 of 15) are wrong
# label = 1 means wrong
# ------------------------------------------------------------
def build_group_target(df):
    grouped = (
        df.groupby("class_id", sort=False)
        .agg(
            n_wrong=("label", "sum"),
            n_resp=("label", "size"),
        )
        .reset_index()
    )

    grouped["p_hat"] = grouped["n_wrong"] / grouped["n_resp"]
    grouped["y"] = (grouped["n_wrong"] >= 8).astype(int)
    return grouped

target_cluster = build_group_target(df_cluster)
target_sample = build_group_target(df_sample)

print("cluster target balance:")
print(target_cluster["y"].value_counts(normalize=True).sort_index())

print("\nsample target balance:")
print(target_sample["y"].value_counts(normalize=True).sort_index())

# ------------------------------------------------------------
# attach class_id in row-block order
# IMPORTANT:
# assumes embeddings were created from df rows in their existing order
# ------------------------------------------------------------
cluster_ids = df_cluster.iloc[::15]["class_id"].reset_index(drop=True)
sample_ids = df_sample.iloc[::15]["class_id"].reset_index(drop=True)

X_cluster = X_cluster.copy()
X_sample = X_sample.copy()

X_cluster["class_id"] = cluster_ids.values
X_sample["class_id"] = sample_ids.values

# merge in targets
X_cluster = X_cluster.merge(
    target_cluster[["class_id", "p_hat", "y"]],
    on="class_id",
    how="left"
)

X_sample = X_sample.merge(
    target_sample[["class_id", "p_hat", "y"]],
    on="class_id",
    how="left"
)

feature_cols = ["H_sem", "D_cos", "D_pair", "n_clusters", "sigma2_S", "D_cos_var"]

Xc = X_cluster[feature_cols].copy()
yc = X_cluster["y"].copy()

Xs = X_sample[feature_cols].copy()
ys = X_sample["y"].copy()

print("final aligned shapes:")
print("Xc:", Xc.shape, "yc:", yc.shape)
print("Xs:", Xs.shape, "ys:", ys.shape)

# ------------------------------------------------------------
# feature agreement: compare matched class-level features
# between sample and cluster
# ------------------------------------------------------------
feature_corr_rows = []

for f in feature_cols:
    pearson = np.corrcoef(X_sample[f], X_cluster[f])[0, 1]
    spearman = X_sample[f].corr(X_cluster[f], method="spearman")

    feature_corr_rows.append({
        "feature": f,
        "pearson": pearson,
        "spearman": spearman,
    })

feature_corr_df = pd.DataFrame(feature_corr_rows).sort_values("pearson", ascending=False)

print("feature agreement: sample vs cluster")
print(feature_corr_df)

# ------------------------------------------------------------
# model: elastic-net logistic regression
# this is intentionally minimal and interpretable
# ------------------------------------------------------------
def make_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            solver="saga",
            l1_ratio=0.5,
            max_iter=5000,
            random_state=0,
        ))
    ])


def evaluate_in_domain(X, y, label, test_size=0.2, random_state=0):
    """
    Train/test within the same dataset.
    Returns:
      summary dict
      fitted model
      X_train, X_test, y_train, y_test
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    cv_aucs = []
    cv_aps = []

    for tr_idx, val_idx in cv.split(X_train, y_train):
        X_tr = X_train.iloc[tr_idx]
        y_tr = y_train.iloc[tr_idx]
        X_val = X_train.iloc[val_idx]
        y_val = y_train.iloc[val_idx]

        model = make_model()
        model.fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_val)[:, 1]

        cv_aucs.append(roc_auc_score(y_val, y_prob))
        cv_aps.append(average_precision_score(y_val, y_prob))

    final_model = make_model()
    final_model.fit(X_train, y_train)
    y_prob_test = final_model.predict_proba(X_test)[:, 1]

    result = {
        "setting": label,
        "cv_auc_mean": np.mean(cv_aucs),
        "cv_auc_std": np.std(cv_aucs),
        "cv_ap_mean": np.mean(cv_aps),
        "cv_ap_std": np.std(cv_aps),
        "test_auc": roc_auc_score(y_test, y_prob_test),
        "test_ap": average_precision_score(y_test, y_prob_test),
    }

    return result, final_model, X_train, X_test, y_train, y_test


def evaluate_transfer(X_train, y_train, X_test, y_test, label):
    """
    Train on one dataset, test on another.
    CV is run on the training dataset only.
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

    cv_aucs = []
    cv_aps = []

    for tr_idx, val_idx in cv.split(X_train, y_train):
        X_tr = X_train.iloc[tr_idx]
        y_tr = y_train.iloc[tr_idx]
        X_val = X_train.iloc[val_idx]
        y_val = y_train.iloc[val_idx]

        model = make_model()
        model.fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_val)[:, 1]

        cv_aucs.append(roc_auc_score(y_val, y_prob))
        cv_aps.append(average_precision_score(y_val, y_prob))

    final_model = make_model()
    final_model.fit(X_train, y_train)
    y_prob_test = final_model.predict_proba(X_test)[:, 1]

    result = {
        "setting": label,
        "cv_auc_mean": np.mean(cv_aucs),
        "cv_auc_std": np.std(cv_aucs),
        "cv_ap_mean": np.mean(cv_aps),
        "cv_ap_std": np.std(cv_aps),
        "test_auc": roc_auc_score(y_test, y_prob_test),
        "test_ap": average_precision_score(y_test, y_prob_test),
    }

    return result, final_model

    # ------------------------------------------------------------
# in-domain: sample -> sample
# ------------------------------------------------------------
res_sample, model_sample, Xs_train, Xs_test, ys_train, ys_test = evaluate_in_domain(
    Xs, ys, label="sample -> sample"
)

# ------------------------------------------------------------
# in-domain: cluster -> cluster
# ------------------------------------------------------------
res_cluster, model_cluster, Xc_train, Xc_test, yc_train, yc_test = evaluate_in_domain(
    Xc, yc, label="cluster -> cluster"
)

in_domain_results = pd.DataFrame([res_sample, res_cluster])
print(in_domain_results)

# ------------------------------------------------------------
# transfer: train on stochastic sample, test on semantic cluster
# THIS IS THE MAIN SURROGATE TEST
# ------------------------------------------------------------
res_transfer, model_transfer = evaluate_transfer(
    X_train=Xs,
    y_train=ys,
    X_test=Xc,
    y_test=yc,
    label="sample -> cluster"
)

res_reverse = evaluate_transfer(
    Xc, yc,
    Xs, ys,
    label="cluster -> sample"
)

transfer_results = pd.DataFrame([res_transfer])
print(transfer_results)

rev_transfer_results = pd.DataFrame([res_reverse])
print(rev_transfer_results)


# ------------------------------------------------------------
# combined summary
# ------------------------------------------------------------
summary_df = pd.concat([in_domain_results, transfer_results], axis=0).reset_index(drop=True)

print(summary_df)

# ------------------------------------------------------------
# interpretation helper
# ------------------------------------------------------------
auc_sample = summary_df.loc[summary_df["setting"] == "sample -> sample", "test_auc"].iloc[0]
auc_cluster = summary_df.loc[summary_df["setting"] == "cluster -> cluster", "test_auc"].iloc[0]
auc_transfer = summary_df.loc[summary_df["setting"] == "sample -> cluster", "test_auc"].iloc[0]

print("Feature agreement summary:")
print(feature_corr_df.to_string(index=False))

print("\nAUC summary:")
print(summary_df[["setting", "test_auc", "test_ap"]].to_string(index=False))

print("\nTransfer preservation ratio:")
print(f"sample->cluster / cluster->cluster = {auc_transfer / auc_cluster:.3f}")

print("\nInterpretation guide:")
print("- If sample->cluster is close to sample->sample, the learned mapping transfers.")
print("- If sample->cluster is also reasonably close to cluster->cluster, stochastic sampling is a good surrogate.")
print("- If sample->cluster collapses far below sample->sample, the surrogate fails.")


CLUSTER DATA
--------------------------------------------------
rows: 15000
unique class_id: 1000
label distribution:
label
0    0.401333
1    0.598667
Name: proportion, dtype: float64

responses per class_id:
count
15    1000
Name: count, dtype: int64

SAMPLE DATA
--------------------------------------------------
rows: 15000
unique class_id: 1000
label distribution:
label
0    0.396133
1    0.603867
Name: proportion, dtype: float64

responses per class_id:
count
15    1000
Name: count, dtype: int64
raw embedding shapes:
E_cluster: (15000, 384)
E_sample:  (15000, 384)

grouped embedding shapes:
E_cluster_grouped: (1000, 15, 384)
E_sample_grouped:  (1000, 15, 384)
feature matrix shapes:
X_cluster: (1000, 6)
X_sample:  (1000, 6)

columns:
['H_sem', 'D_cos', 'D_pair', 'n_clusters', 'sigma2_S', 'D_cos_var']
feature matrix shapes:
X_cluster: (1000, 6)
X_sample:  (1000, 6)

columns:
['H_sem', 'D_cos', 'D_pair', 'n_clusters', 'sigma2_S', 'D_cos_var']
cluster target balance:
y
0    0.404
1  